# Commodity-Trend Model vs. 60/40

Presentation notebook for the completed analysis. This notebook loads prepared outputs and delegates plotting, table formatting, and lightweight display analytics to `trend_analysis` modules.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "trend_analysis").exists() else Path.cwd() / "Notebooks"
sys.path.insert(0, str(NOTEBOOK_DIR))

from trend_analysis.config import BENCHMARK_6040, BONDS, COMMODITY_TREND, DYNAMIC_MV, EQUITY, SLEEVE_9010, STATIC_MV
from trend_analysis.data import (
    expected_output_files,
    load_core_summary,
    load_drawdown_buckets,
    load_drawdown_episodes,
    load_dynamic_weights,
    load_five_strategy_summary,
    load_merged_daily,
    load_part5_calendar,
    load_part5_conditional,
    load_part6_conditional,
    load_part6_counts,
    load_part6_stress,
    load_quantile_windows,
    load_sleeve_summary,
    load_static_mv_weights,
    prepare_core_and_strategy_returns,
)
from trend_analysis.original_style import (
    display_evaluation_block,
    display_metric_table,
    dual_beta_regression_table,
    format_metric_table,
    monthly_return_heatmap_original,
    plot_annual_return_bars,
    plot_benchmark_drawdown,
    plot_core_daily_returns,
    plot_core_nav,
    plot_correlation_combo_original,
    plot_drawdown_buckets_original,
    plot_drawdown_episode_bars_original,
    plot_drawdown_episode_timeline_original,
    plot_drawdown_original,
    plot_dynamic_weight_step,
    plot_mean_variance_frontier_original,
    plot_nav_original,
    plot_part5_active_return_boxplots_original,
    plot_part6_active_return_boxplots_original,
    plot_part6_metric_panel_original,
    plot_quintile_matched_bars_original,
    plot_relative_wealth_original,
    plot_return_scatter_original,
    plot_stacked_annual_returns_original,
    plot_stacked_drawdowns_original,
    plot_stock_bond_corr_original,
    plot_stress_matched_panels_original,
    style_regression_table,
)
from trend_analysis.plots import apply_theme
from trend_analysis.tables import presentation_table, style_summary_table

apply_theme()

## Data Inputs

Expects the analysis CSVs in `Notebooks/output/` to already exist.

In [ ]:
missing = [path.name for path in expected_output_files() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing presentation inputs: {missing}")

merged = load_merged_daily()
returns, core_returns, strategy_returns = prepare_core_and_strategy_returns()
core_summary = load_core_summary()
sleeve_summary = load_sleeve_summary()
five_strategy_summary = load_five_strategy_summary()
static_mv_weights = load_static_mv_weights()
dynamic_weights = load_dynamic_weights()
episodes = load_drawdown_episodes()
quantile_windows = load_quantile_windows()
drawdown_buckets = load_drawdown_buckets()
part5_calendar = load_part5_calendar()
part5_conditional = load_part5_conditional()
part6_counts = load_part6_counts()
part6_conditional = load_part6_conditional()
part6_stress = load_part6_stress()

## Part 1 — Core Performance

The Commodity-Trend Model and the 60/40 benchmark are shown on the same daily calendar using growth of $1, daily returns, drawdowns, and benchmark-only calendar context.

In [ ]:
display_metric_table(core_summary)

Commodity-Trend earned 4.38% annualized vs 7.67% for 60/40 from 2002 to 2026, ending at $283.4M vs $602.5M on a $100M start. Risk was lower across the board: 7.08% vol vs 10.87%, and a -15.93% max drawdown vs -32.68%.

In [ ]:
plot_core_nav(merged)
plot_core_daily_returns(core_returns)

60/40 compounded more wealth ($602.5M vs $283.4M ending NAV on a $100M start), but Commodity-Trend's path was smoother. Worst daily loss was -3.86% vs -6.52% for 60/40.

In [ ]:
display(monthly_return_heatmap_original(core_returns[BENCHMARK_6040]))
plot_benchmark_drawdown(core_returns)
plot_annual_return_bars(core_returns[BENCHMARK_6040], "60/40 Annual Returns")

60/40's worst calendar years were 2008 (-18.3%) and 2022 (-15.3%). The worst single months were October 2008 (-9.9%), September 2022 (-6.9%), and February 2009 (-6.6%).

## Part 2 — Correlation

The relationship between the Commodity-Trend Model and 60/40 is summarized with daily-return scatter, rolling correlation, and calendar-year correlation.

In [ ]:
plot_return_scatter_original(core_returns)
plot_correlation_combo_original(core_returns)

Full-sample daily correlation with 60/40 is 0.16. The 252-day rolling correlation ranged from -0.21 to 0.64, and calendar-year correlation went negative in several years (2003: -0.17, 2008: -0.04), so the diversification benefit is real but regime-dependent.

## Part 3 — Diversification

The portfolio exhibits compare standalone 60/40, standalone Commodity-Trend exposure, a 90/10 blend, and long-only mean-variance allocations.

In [ ]:
display_metric_table(sleeve_summary)
display_evaluation_block(SLEEVE_9010, strategy_returns[SLEEVE_9010], weight=0.10)
plot_nav_original(
    {BENCHMARK_6040: strategy_returns[BENCHMARK_6040], SLEEVE_9010: strategy_returns[SLEEVE_9010], COMMODITY_TREND: strategy_returns[COMMODITY_TREND]},
    "90/10 Blend: NAV index",
)
plot_drawdown_original(
    {BENCHMARK_6040: strategy_returns[BENCHMARK_6040], SLEEVE_9010: strategy_returns[SLEEVE_9010], COMMODITY_TREND: strategy_returns[COMMODITY_TREND]},
    "90/10 Blend: Drawdown",
)

A 10% Commodity-Trend sleeve raises Sharpe from 0.71 to 0.75, lowers vol from 10.87% to 9.92%, and improves max drawdown from -32.68% to -29.73%. Annual return falls slightly (7.67% to 7.41%), with ending NAV at $567.9M vs $602.5M on a $100M start.

In [ ]:
display_metric_table(five_strategy_summary)
plot_nav_original(
    {name: strategy_returns[name] for name in [BENCHMARK_6040, COMMODITY_TREND, SLEEVE_9010, STATIC_MV, DYNAMIC_MV]},
    "Five strategies: NAV",
)
plot_stacked_drawdowns_original(strategy_returns)
plot_stacked_annual_returns_original(strategy_returns)

Sharpe improves with larger trend weights: 0.71 (60/40), 0.75 (90/10 blend), 0.90 (static MV), 0.93 (dynamic MV). Dynamic MV earned 7.36% annualized vs 7.67% for 60/40, with 7.95% vol vs 10.87% and a -18.33% max drawdown vs -32.68%.

In [ ]:
display_metric_table(static_mv_weights)
plot_mean_variance_frontier_original(core_returns, static_mv_weights)
plot_dynamic_weight_step(dynamic_weights)
display_evaluation_block(STATIC_MV, strategy_returns[STATIC_MV], weight=float(static_mv_weights.loc[static_mv_weights["Strategy"].str.contains("Tangency", case=False), "w_T"].iloc[0]))
display_evaluation_block(DYNAMIC_MV, strategy_returns[DYNAMIC_MV])

The static tangency portfolio holds 56% Commodity-Trend / 44% 60/40 (Sharpe 0.88); the minimum-variance portfolio holds 73.7% Commodity-Trend. The dynamic tangency weight ranged from 0% to 100% over the sample and ended at 41.8% in April 2026, so the optimal weight is very sensitive to the historical window.

## Part 4 — 60/40 Drawdowns

These exhibits focus on how the Commodity-Trend Model behaved when the 60/40 benchmark was under pressure.

In [ ]:
display(style_summary_table(presentation_table(episodes)))
plot_drawdown_episode_timeline_original(core_returns, episodes)
plot_drawdown_episode_bars_original(episodes)

In the two largest 60/40 drawdowns, Commodity-Trend gained while 60/40 lost: 2007-2009 (+4.9% vs -32.4%) and 2022 (+3.0% vs -19.2%). It did not offset the shorter 2020 COVID drawdown (-0.7% vs -19.3%) or the 2025 drawdown (-4.1% vs -10.7%).

In [ ]:
display(style_summary_table(presentation_table(quantile_windows)))
plot_stress_matched_panels_original(quantile_windows, q_use=0.10)
plot_quintile_matched_bars_original(core_returns)
display(style_summary_table(presentation_table(drawdown_buckets)))
plot_drawdown_buckets_original(drawdown_buckets)

In the worst 10% of 21-day 60/40 windows, 60/40 averaged -4.76% and Commodity-Trend -0.44% (+4.32pp active). In the worst 10% of 63-day windows, 60/40 averaged -6.94% and Commodity-Trend +0.21% (+7.15pp). Caveat from the drawdown-depth buckets: when 60/40 was already more than 20% below its prior high, Commodity-Trend's annualized return was -12.0%, so the model damps but does not invert deep stress.

### Asymmetric Capture Regression

Formula:

`r_Commodity-Trend = alpha + beta_up * r_60/40 * 1{r_60/40 > 0} + beta_down * r_60/40 * 1{r_60/40 < 0} + error`

Standard errors are HC1 robust.

In [ ]:
display(style_regression_table(dual_beta_regression_table(core_returns)))

Up-day beta to 60/40 is 0.073 and down-day beta is 0.136. Both are small, so Commodity-Trend's return path is not just a scaled-down 60/40 in either direction.

## Part 5 — Equity and Bond Sleeves

Calendar-horizon active returns summarize when Commodity-Trend outperformed equity and bond sleeves.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.ticker import FuncFormatter

from trend_analysis.original_style import STRATEGY_COLORS, _style_axes

TABLE_COLUMNS = [
    "horizon",
    "comparator",
    "condition",
    "n_windows",
    "outright_win_rate",
    "risk_adjusted_win_rate",
    "average_active_return",
    "median_active_return",
    "avg_trend_max_dd",
    "avg_comparator_max_dd",
    "avg_60_40_max_dd",
]

TABLE_RENAMES = {
    "horizon": "Horizon",
    "comparator": "Comparator",
    "condition": "Condition",
    "n_windows": "N windows",
    "outright_win_rate": "Outright win rate",
    "risk_adjusted_win_rate": "Risk-adjusted win rate",
    "trend_win_rate_vs_60_40": "Outright win rate",
    "avg_active_return_t_minus_6040": "Mean active return",
    "median_active_return_t_minus_6040": "Median active return",
    "average_active_return": "Mean active return",
    "median_active_return": "Median active return",
    "avg_trend_max_dd": "Avg trend max DD",
    "avg_comparator_max_dd": "Avg comparator max DD",
    "avg_60_40_max_dd": "Avg 60/40 max DD",
}

GROUP_COLORS = {
    "Month": "#EAF2F8",
    "Quarter": "#F3EAF8",
    "Year": "#EAF6EF",
}


def _title_value(value):
    if not isinstance(value, str):
        return value
    return value.replace("_", " ").title()


def _window_max_drawdown(series):
    growth = (1.0 + series.dropna()).cumprod()
    if growth.empty:
        return np.nan
    return (growth / growth.cummax() - 1.0).min()


def add_part5_drawdown_columns(summary_df, return_df):
    freq_map = {"month": "ME", "quarter": "QE", "year": "YE"}
    out = summary_df.copy()
    trend_dd = []
    comparator_dd = []
    for _, row in out.iterrows():
        horizon = str(row["horizon"]).lower()
        comparator = row["comparator"]
        freq = freq_map[horizon]
        needed = list(dict.fromkeys([COMMODITY_TREND, comparator, EQUITY, BONDS]))
        window_returns = (1.0 + return_df[needed].dropna(how="any")).resample(freq).prod() - 1.0
        max_dd = return_df[[COMMODITY_TREND, comparator]].dropna(how="any").resample(freq).agg(_window_max_drawdown)

        condition = str(row.get("condition", "")).lower()
        mask = pd.Series(True, index=window_returns.index)
        if "equity_return < 0" in condition:
            mask &= window_returns[EQUITY].lt(0)
        if "bond_return < 0" in condition:
            mask &= window_returns[BONDS].lt(0)

        selected = max_dd.reindex(window_returns.index[mask])
        trend_dd.append(selected[COMMODITY_TREND].mean())
        comparator_dd.append(selected[comparator].mean())

    out["avg_trend_max_dd"] = trend_dd
    out["avg_comparator_max_dd"] = comparator_dd
    return out


def compact_outperformance_table(df, default_condition="All windows"):
    out = df.copy()
    if "condition" not in out.columns:
        out["condition"] = default_condition
    cols = [col for col in TABLE_COLUMNS if col in out.columns]
    out = out[cols].rename(columns=TABLE_RENAMES)
    for col in ["Horizon", "Comparator", "Condition"]:
        if col in out.columns:
            out[col] = out[col].map(_title_value)
    return out


def compact_part6_table(df, include_comparator=True):
    out = df.copy()
    out["comparator"] = BENCHMARK_6040
    cols = [
        "horizon",
        "comparator",
        "condition",
        "n_windows",
        "trend_win_rate_vs_60_40",
        "avg_active_return_t_minus_6040",
        "median_active_return_t_minus_6040",
        "avg_trend_max_dd",
        "avg_60_40_max_dd",
    ]
    out = out[[col for col in cols if col in out.columns]].rename(columns=TABLE_RENAMES)
    if not include_comparator and "Comparator" in out.columns:
        out = out.drop(columns="Comparator")
    for col in ["Horizon", "Comparator", "Condition"]:
        if col in out.columns:
            out[col] = out[col].map(_title_value)
    return out


def compact_part6_counts_table(df):
    out = df[["horizon", "n_windows", "positive_corr_windows", "positive_corr_share", "avg_eq_bond_corr_positive"]].copy()
    out = out.rename(
        columns={
            "horizon": "Horizon",
            "n_windows": "N windows",
            "positive_corr_windows": "Positive-corr windows",
            "positive_corr_share": "Positive-corr share",
            "avg_eq_bond_corr_positive": "Mean corr when positive",
        }
    )
    out["Horizon"] = out["Horizon"].map(_title_value)
    return out


def style_grouped_extremes(df):
    pct_cols = [
        col
        for col in df.columns
        if "rate" in col.lower()
        or "return" in col.lower()
        or "share" in col.lower()
        or "corr" in col.lower()
        or "drawdown" in col.lower()
        or " dd" in col.lower()
    ]
    formats = {col: "{:.1%}" for col in pct_cols}
    if "N windows" in df.columns:
        formats["N windows"] = "{:,.0f}"
    if "Positive-corr windows" in df.columns:
        formats["Positive-corr windows"] = "{:,.0f}"

    def row_styles(row):
        color = GROUP_COLORS.get(row.get("Horizon"), "#FFFFFF")
        styles = [f"border-top: 2px solid {color};"] * len(row)
        if "Condition" in row.index:
            styles[row.index.get_loc("Condition")] += f" background-color: {color}; font-weight: 600;"
        elif "Horizon" in row.index:
            styles[row.index.get_loc("Horizon")] += f" background-color: {color}; font-weight: 600;"
        return styles

    styler = df.style.format(formats, na_rep="").apply(row_styles, axis=1)
    return (
        styler.set_properties(**{"text-align": "center"})
        .set_table_styles(
            [
                {"selector": "th", "props": [("text-align", "center"), ("font-weight", "600"), ("background-color", "#F7F9FC")]},
                {"selector": "td", "props": [("border-bottom", "1px solid #E6EAF0")]},
            ]
        )
        .hide(axis="index")
    )


part5_calendar_with_dd = add_part5_drawdown_columns(part5_calendar, returns)
part5_conditional_with_dd = add_part5_drawdown_columns(part5_conditional, returns)
part5_calendar_compact = compact_outperformance_table(part5_calendar_with_dd).drop(columns="Condition")
part5_conditional_compact = compact_outperformance_table(part5_conditional_with_dd).sort_values(["Horizon", "Condition", "Comparator"])

display(style_grouped_extremes(part5_calendar_compact))
display(style_grouped_extremes(part5_conditional_compact))
plot_part5_active_return_boxplots_original(returns)

Commodity-Trend beats equities only 38.1% of months, 34.0% of quarters, and 20.8% of years. Hit rates jump in down-equity windows: 83.5% of months (+3.50pp) and 92.3% of quarters (+8.31pp). Versus bonds it wins 50.5% of months overall and 72.9% in down-bond months (+1.08pp).

## Part 6 — Positive Stock-Bond Correlation

This section isolates environments where traditional stock-bond diversification was weaker and compares Commodity-Trend against 60/40.

In [ ]:
part6_counts_compact = compact_part6_counts_table(part6_counts)
part6_conditional_compact = compact_part6_table(part6_conditional).sort_values(["Horizon", "Condition"])
part6_stress_compact = compact_part6_table(part6_stress).sort_values(["Horizon", "Condition"])

display(style_grouped_extremes(part6_counts_compact))
display(style_grouped_extremes(part6_conditional_compact))
display(style_grouped_extremes(part6_stress_compact))

Stock-bond correlation was positive in 24.7% of months (72 obs) and 20.6% of quarters (20 obs). Across all positive-correlation months, Commodity-Trend beat 60/40 only 40.3% of the time (-0.34pp). The win rate rises to 69.6% (+1.24pp) when those months were also down for 60/40.

In [ ]:
def _pct_axis(ax):
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:.0%}"))


def plot_part6_metric_panel_with_counts(df, title_context, condition=None):
    if condition is None:
        plot_df = df[df["regime"].eq("Positive stock-bond corr")].copy()
    else:
        plot_df = df[df["condition"].eq(condition)].copy()
    plot_df["horizon"] = plot_df["horizon"].str.title()
    plot_df["x_label"] = plot_df["horizon"] + "\nN=" + plot_df["n_windows"].astype(int).astype(str)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.9))
    specs = [
        ("avg_trend_return", "avg_60_40_return", "Avg return"),
        ("avg_trend_sharpe", "avg_60_40_sharpe", "Avg Sharpe"),
        ("avg_trend_max_dd", "avg_60_40_max_dd", "Avg max drawdown"),
    ]
    for ax, (trend_col, bench_col, ylabel) in zip(axes, specs):
        x = np.arange(len(plot_df))
        ax.bar(x - 0.18, plot_df[bench_col], width=0.36, color=STRATEGY_COLORS[BENCHMARK_6040], label="60/40")
        ax.bar(x + 0.18, plot_df[trend_col], width=0.36, color=STRATEGY_COLORS[COMMODITY_TREND], label=COMMODITY_TREND)
        ax.set_xticks(x)
        ax.set_xticklabels(plot_df["x_label"])
        ax.axhline(0, color="#333333", linewidth=0.8)
        ax.set_ylabel(ylabel)
        if "return" in trend_col or "dd" in trend_col:
            _pct_axis(ax)
        _style_axes(ax, "y")
    fig.suptitle(f"Commodity-Trend vs 60/40 when {title_context}", fontsize=13, fontweight="600", y=1.02)
    axes[0].legend(loc="best")
    plt.tight_layout()
    return fig


plot_stock_bond_corr_original(returns.dropna(subset=[EQUITY, BONDS]))
plot_part6_metric_panel_with_counts(part6_conditional, "stock-bond correlation is positive")
plot_part6_metric_panel_with_counts(
    part6_stress,
    "stock-bond correlation is positive and stocks are down",
    condition="positive corr and equity_return < 0",
)
plot_part6_active_return_boxplots_original(returns)

Across all positive-correlation windows, Commodity-Trend lagged 60/40 by -0.34pp per month, -1.39pp per quarter, and -6.93pp per year. The stress overlap is what matters: in positive-correlation 60/40-down months it won 69.6% (+1.24pp), and when stocks and bonds were both down it won 77.8% (+1.65pp).

## Portfolio Takeaway

Commodity-Trend works as a diversifying sleeve inside 60/40, not as a substitute. The value comes from low (and time-varying) correlation, smaller losses in the largest 60/40 drawdowns, and Sharpe improvement when added at conservative weights.